# ACPDS paper reproduction on Kaggle

Reproduce the official baseline from:

> Martin Marek, **Image-Based Parking Space Occupancy Classification: Dataset and Baseline** (2021)

- Paper: https://arxiv.org/abs/2107.12207
- Official code: https://github.com/martin-marek/parking-space-occupancy

This notebook uses the authors' official ACPDS split, model implementation, dataset, and pretrained weights.

## Reproduction levels

1. Inspect the ACPDS dataset and official train/validation/test split.
2. Evaluate the authors' pretrained `RCNN-128-square` model.
3. Run a short five-epoch smoke training.
4. Optionally run the complete 100-epoch single-model reproduction.

The complete paper sweep in `train_all_models.py` is intentionally not run: it trains 12 configurations five times each and is much more expensive than reproducing one baseline.

## Kaggle settings

- Accelerator: **GPU T4**
- Internet: **On**
- Run cells from top to bottom.


In [ ]:
# Configuration — edit only this cell when needed
SEED = 42

RUN_PRETRAINED_EVALUATION = True
RUN_SMOKE_TRAINING = True
SMOKE_EPOCHS = 5

# Change this to True only after the smoke run succeeds.
RUN_FULL_TRAINING = False
FULL_EPOCHS = 100

MODEL_ROI_RESOLUTION = 128
POOLING_TYPE = "square"

print("Configuration loaded.")
print(f"Smoke training: {RUN_SMOKE_TRAINING} ({SMOKE_EPOCHS} epochs)")
print(f"Full training:  {RUN_FULL_TRAINING} ({FULL_EPOCHS} epochs)")

## 1. Check the Kaggle environment

The current Kaggle PyTorch build requires a GPU with a supported CUDA compute capability. A T4 is suitable. If Kaggle assigns a P100 and the cell reports an incompatibility, switch the accelerator/session to T4 before continuing.


In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import torchvision
from matplotlib.patches import Polygon
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. In Kaggle, choose Settings > Accelerator > GPU T4.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
print("GPU:", gpu_name)
print("CUDA capability:", gpu_capability)

if gpu_capability[0] < 7:
    raise RuntimeError(
        f"{gpu_name} has CUDA capability {gpu_capability}. "
        "Use a Kaggle T4 session with the current PyTorch build."
    )

## 2. Clone the official repository

In [ ]:
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "parking-space-occupancy"
DATA_DIR = REPO_DIR / "dataset" / "data"
OUTPUT_ROOT = WORK_ROOT / "acpds_reproduction_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/martin-marek/parking-space-occupancy.git",
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    print("Repository already exists; clone skipped.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

OFFICIAL_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()

print("Repository:", REPO_DIR)
print("Official commit:", OFFICIAL_COMMIT)
print("Top-level files:")
for path in sorted(REPO_DIR.iterdir()):
    print(" -", path.name)

## 3. Apply a minimal Torchvision compatibility update

The 2021 repository uses the deprecated `pretrained=True` argument. This cell updates only that API call to the current `weights=ResNet50_Weights.DEFAULT` form. The ResNet-50 architecture and ImageNet initialization remain unchanged.


In [ ]:
rcnn_file = REPO_DIR / "models" / "rcnn.py"
rcnn_source = rcnn_file.read_text(encoding="utf-8")

old_import = "from torchvision.models import resnet50"
new_import = "from torchvision.models import ResNet50_Weights, resnet50"
old_call = "resnet50(pretrained=True, norm_layer=FrozenBatchNorm2d)"
new_call = "resnet50(weights=ResNet50_Weights.DEFAULT, norm_layer=FrozenBatchNorm2d)"

changed = False
if old_import in rcnn_source:
    rcnn_source = rcnn_source.replace(old_import, new_import)
    changed = True
if old_call in rcnn_source:
    rcnn_source = rcnn_source.replace(old_call, new_call)
    changed = True

if changed:
    rcnn_file.write_text(rcnn_source, encoding="utf-8")
    print("Applied Torchvision compatibility update.")
else:
    print("Compatibility update already applied or not required.")

assert new_import in rcnn_file.read_text(encoding="utf-8")
assert new_call in rcnn_file.read_text(encoding="utf-8")

## 4. Download and extract the official ACPDS dataset

In [ ]:
DATASET_URL = (
    "https://pub-e8bbdcbe8f6243b2a9933704a9b1d8bc.r2.dev/"
    "parking%2Frois_gopro.zip"
)
DATASET_ZIP = WORK_ROOT / "rois_gopro.zip"


def download_file(url: str, destination: Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with temporary.open("wb") as file, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))
    temporary.replace(destination)


annotations_file = DATA_DIR / "annotations.json"

if not annotations_file.exists():
    if not DATASET_ZIP.exists() or not zipfile.is_zipfile(DATASET_ZIP):
        if DATASET_ZIP.exists():
            DATASET_ZIP.unlink()
        print("Downloading the official ACPDS dataset...")
        download_file(DATASET_URL, DATASET_ZIP)

    if not zipfile.is_zipfile(DATASET_ZIP):
        raise RuntimeError("The downloaded dataset is not a valid ZIP file.")

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Extracting dataset...")
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        archive.extractall(DATA_DIR)
else:
    print("Dataset is already extracted; download skipped.")

if not annotations_file.exists():
    raise FileNotFoundError(
        f"Expected {annotations_file}, but it was not found after extraction."
    )

print("Dataset directory:", DATA_DIR)
print("Dataset ZIP size: {:.1f} MB".format(DATASET_ZIP.stat().st_size / 1024**2))

## 5. Validate the official split and class balance

In [ ]:
with annotations_file.open("r", encoding="utf-8") as file:
    annotations = json.load(file)

image_dir = DATA_DIR / "images"
image_files = sorted(image_dir.glob("*"))

rows = []
for split_name in ["train", "valid", "test"]:
    split = annotations[split_name]
    labels = [label for image_labels in split["occupancy_list"] for label in image_labels]
    occupied = int(sum(labels))
    total_spaces = len(labels)
    rows.append(
        {
            "split": split_name,
            "images": len(split["file_names"]),
            "parking_spaces": total_spaces,
            "vacant": total_spaces - occupied,
            "occupied": occupied,
            "occupied_ratio": occupied / total_spaces if total_spaces else 0,
        }
    )

split_table = pd.DataFrame(rows)
display(split_table)

missing_images = []
for split_name in ["train", "valid", "test"]:
    for filename in annotations[split_name]["file_names"]:
        if not (image_dir / filename).exists():
            missing_images.append(filename)

print("Extracted image files:", len(image_files))
print("Missing referenced images:", len(missing_images))

if missing_images:
    raise FileNotFoundError(f"Missing dataset images, for example: {missing_images[:5]}")

## 6. Visualize one annotated image

In [ ]:
sample_split = "train"
sample_index = 0
sample_annotations = annotations[sample_split]
sample_filename = sample_annotations["file_names"][sample_index]
sample_rois = np.asarray(sample_annotations["rois_list"][sample_index])
sample_labels = np.asarray(sample_annotations["occupancy_list"][sample_index])

sample_image_tensor = torchvision.io.read_image(str(image_dir / sample_filename))
sample_image = sample_image_tensor.permute(1, 2, 0).numpy()
height, width = sample_image.shape[:2]

fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(sample_image)

for roi, label in zip(sample_rois, sample_labels):
    polygon = roi.copy()
    polygon[:, 0] *= width
    polygon[:, 1] *= height
    color = "red" if int(label) == 1 else "lime"
    ax.add_patch(Polygon(polygon, closed=True, fill=False, edgecolor=color, linewidth=1.5))

ax.set_title(
    f"{sample_split}: {sample_filename} | "
    f"red=occupied, green=vacant | spaces={len(sample_labels)}"
)
ax.axis("off")
plt.show()

## 7. Load the official dataset and model code

In [ ]:
from dataset import acpds
from models.rcnn import RCNN
from utils import transforms
from utils.engine import eval_one_epoch, train_model

train_ds, valid_ds, test_ds = acpds.create_datasets(str(DATA_DIR))

print("Training images:", len(train_ds.dataset))
print("Validation images:", len(valid_ds.dataset))
print("Test images:", len(test_ds.dataset))
print("Device:", device)

## 8. Evaluate the authors' pretrained model

In [ ]:
WEIGHTS_URL = (
    "https://pub-e8bbdcbe8f6243b2a9933704a9b1d8bc.r2.dev/"
    "parking%2FRCNN_128_square_gopro.pt"
)
WEIGHTS_PATH = WORK_ROOT / "RCNN_128_square_gopro.pt"

if not WEIGHTS_PATH.exists():
    print("Downloading official pretrained weights...")
    download_file(WEIGHTS_URL, WEIGHTS_PATH)
else:
    print("Pretrained weights already exist; download skipped.")

print("Weights size: {:.1f} MB".format(WEIGHTS_PATH.stat().st_size / 1024**2))

pretrained_model = RCNN(
    roi_res=MODEL_ROI_RESOLUTION,
    pooling_type=POOLING_TYPE,
).to(device)

try:
    state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
except TypeError:
    state_dict = torch.load(WEIGHTS_PATH, map_location=device)

pretrained_model.load_state_dict(state_dict)
pretrained_model.eval()

pretrained_loss = None
pretrained_accuracy = None

if RUN_PRETRAINED_EVALUATION:
    started = time.time()
    pretrained_loss, pretrained_accuracy = eval_one_epoch(
        pretrained_model,
        test_ds,
        res=None,
    )
    pretrained_seconds = time.time() - started
    print(f"Test loss: {pretrained_loss:.4f}")
    print(f"Test accuracy: {pretrained_accuracy:.4%}")
    print(f"Evaluation time: {pretrained_seconds:.1f} seconds")
else:
    print("Pretrained evaluation skipped by configuration.")

## 9. Visualize pretrained predictions

In [ ]:
test_index = 0
test_image, test_rois, test_labels = test_ds.dataset[test_index]

with torch.inference_mode():
    processed_image = transforms.preprocess(test_image).to(device)
    logits = pretrained_model(processed_image, test_rois.to(device))
    probabilities = logits.softmax(dim=1)[:, 1].cpu()
    predictions = (probabilities >= 0.5).to(torch.int64)

display_image = test_image.permute(1, 2, 0).numpy()
height, width = display_image.shape[:2]

fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(display_image)

for roi, prediction, target, probability in zip(
    test_rois.numpy(),
    predictions.numpy(),
    test_labels.numpy(),
    probabilities.numpy(),
):
    polygon = roi.copy()
    polygon[:, 0] *= width
    polygon[:, 1] *= height

    correct = int(prediction) == int(target)
    color = ("red" if int(prediction) == 1 else "lime") if correct else "yellow"
    ax.add_patch(Polygon(polygon, closed=True, fill=False, edgecolor=color, linewidth=1.5))

    center_x = polygon[:, 0].mean()
    center_y = polygon[:, 1].mean()
    ax.text(
        center_x,
        center_y,
        f"{probability:.2f}",
        color="black",
        fontsize=7,
        ha="center",
        va="center",
        bbox={"facecolor": color, "alpha": 0.65, "edgecolor": "none", "pad": 1},
    )

image_accuracy = float((predictions == test_labels).float().mean())
ax.set_title(
    f"Pretrained predictions | green=vacant, red=occupied, yellow=wrong | "
    f"image accuracy={image_accuracy:.2%}"
)
ax.axis("off")
plt.show()

## 10. Five-epoch smoke training

This verifies that loading, augmentation, forward propagation, backpropagation, logging, checkpoint saving, and test evaluation all work. Five epochs are **not** expected to reproduce the final paper accuracy.


In [ ]:
SMOKE_OUTPUT_DIR = OUTPUT_ROOT / "rcnn_128_square_smoke"

if RUN_SMOKE_TRAINING:
    smoke_model = RCNN(
        roi_res=MODEL_ROI_RESOLUTION,
        pooling_type=POOLING_TYPE,
    )

    started = time.time()
    train_model(
        smoke_model,
        train_ds,
        valid_ds,
        test_ds,
        str(SMOKE_OUTPUT_DIR),
        device,
        lr=1e-4,
        epochs=SMOKE_EPOCHS,
        lr_decay=50,
        res=None,
        verbose=True,
    )
    smoke_training_seconds = time.time() - started
    print(f"Smoke training completed in {smoke_training_seconds / 60:.1f} minutes.")
    print("Output:", SMOKE_OUTPUT_DIR)
else:
    print("Smoke training skipped by configuration.")

## 11. Inspect smoke-training results

In [ ]:
smoke_test_metrics = None
smoke_log_file = SMOKE_OUTPUT_DIR / "train_log.csv"
smoke_test_file = SMOKE_OUTPUT_DIR / "test_logs.json"

if smoke_log_file.exists():
    smoke_log = pd.read_csv(smoke_log_file)
    display(smoke_log)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(smoke_log.index + 1, smoke_log["train_loss"], label="train")
    axes[0].plot(smoke_log.index + 1, smoke_log["valid_loss"], label="validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(smoke_log.index + 1, smoke_log["train_accuracy"], label="train")
    axes[1].plot(smoke_log.index + 1, smoke_log["valid_accuracy"], label="validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    plt.show()

if smoke_test_file.exists():
    with smoke_test_file.open("r", encoding="utf-8") as file:
        smoke_test_metrics = json.load(file)
    print("Smoke test metrics:", smoke_test_metrics)
elif RUN_SMOKE_TRAINING:
    raise FileNotFoundError("Smoke training finished without producing test_logs.json")
else:
    print("No smoke-training output is available.")

## 12. Optional complete 100-epoch single-model reproduction

Return to the configuration cell, set `RUN_FULL_TRAINING = True`, and rerun this cell only after the five-epoch smoke training succeeds.

This reproduces one official `RCNN-128-square` baseline. It does not execute the authors' complete 60-run model sweep.


In [ ]:
FULL_OUTPUT_DIR = OUTPUT_ROOT / "rcnn_128_square_full"

if RUN_FULL_TRAINING:
    full_model = RCNN(
        roi_res=MODEL_ROI_RESOLUTION,
        pooling_type=POOLING_TYPE,
    )

    started = time.time()
    train_model(
        full_model,
        train_ds,
        valid_ds,
        test_ds,
        str(FULL_OUTPUT_DIR),
        device,
        lr=1e-4,
        epochs=FULL_EPOCHS,
        lr_decay=50,
        res=None,
        verbose=True,
    )
    full_training_seconds = time.time() - started
    print(f"Full training completed in {full_training_seconds / 3600:.2f} hours.")
    print("Output:", FULL_OUTPUT_DIR)
else:
    print("Full training is disabled. Set RUN_FULL_TRAINING=True when ready.")

## 13. Build the reproduction summary

In [ ]:
full_test_metrics = None
full_test_file = FULL_OUTPUT_DIR / "test_logs.json"
if full_test_file.exists():
    with full_test_file.open("r", encoding="utf-8") as file:
        full_test_metrics = json.load(file)

summary = {
    "paper": "Image-Based Parking Space Occupancy Classification: Dataset and Baseline",
    "official_repository": "https://github.com/martin-marek/parking-space-occupancy",
    "official_commit": OFFICIAL_COMMIT,
    "dataset": "ACPDS / Action-Camera Parking Dataset",
    "model": f"RCNN_{MODEL_ROI_RESOLUTION}_{POOLING_TYPE}",
    "seed": SEED,
    "environment": {
        "python": sys.version.split()[0],
        "pytorch": torch.__version__,
        "torchvision": torchvision.__version__,
        "gpu": gpu_name,
        "cuda_capability": list(gpu_capability),
    },
    "pretrained_test": {
        "loss": pretrained_loss,
        "accuracy": pretrained_accuracy,
    },
    "smoke_training": {
        "epochs": SMOKE_EPOCHS if smoke_test_metrics is not None else None,
        "test": smoke_test_metrics,
    },
    "full_training": {
        "epochs": FULL_EPOCHS if full_test_metrics is not None else None,
        "test": full_test_metrics,
    },
    "paper_reported_baseline": "approximately 98% accuracy on unseen parking lots",
}

summary_path = OUTPUT_ROOT / "reproduction_summary.json"
with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print(json.dumps(summary, indent=2))
print("Saved:", summary_path)

## 14. Package results for download

In [ ]:
archive_base = WORK_ROOT / "acpds_reproduction_results"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", OUTPUT_ROOT))

print("Results archive:", archive_path)
print("Archive size: {:.1f} MB".format(archive_path.stat().st_size / 1024**2))

try:
    from IPython.display import FileLink, display

    display(FileLink(str(archive_path)))
except Exception:
    print("Download the ZIP from the Kaggle file browser under /kaggle/working/.")

## How to judge whether reproduction succeeded

- **Environment success:** Kaggle detects a compatible NVIDIA GPU.
- **Dataset success:** no referenced images are missing and the three official splits are displayed.
- **Inference success:** the official pretrained model completes test evaluation and produces a parking-space visualization.
- **Pipeline success:** the five-epoch smoke run creates `train_log.csv`, `test_logs.json`, and `weights_last_epoch.pt`.
- **Full reproduction:** the 100-epoch run uses the official split and produces a test result close to the result reported by the paper. Exact equality is not required because training randomness, CUDA, PyTorch, and Torchvision versions differ from the 2021 environment.

Keep the JSON summary, training log, test log, final weights, and one prediction image as experiment evidence for the capstone repository.
